<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 6 — The Portfolio Competition

You start a fund with **\$10,000**. Everyone in the class uses the same model (Studio 5: Ms. Womack plus
business rules). What makes your fund different is **your choices**:

1. **Your stocks:** exactly 3 from each of the 11 sectors of the S&P 500 (33 stocks), picked with a filter or by hand.
2. **Your risk tolerance:** how close to the risk ceiling you want to go.
3. **Your fund's name.** Make it a good one: it goes on the leaderboard.

### The rules (the same for everyone)

| Rule | |
|---|---|
| Stocks | exactly 3 from each of the 11 sectors (33 in total) |
| Every stock you hold | at least 2% and at most 10% of the money |
| All of the money invested | yes |
| Risk ceiling | no riskier than the S&P 500 itself over the same past: monthly variance $\le$ 0.00223 |

### How the winner is chosen

You build your fund on **48 months of history (October 2021 to September 2025)**. Dr. Dave scores every fund on
**the next 12 months (October 2025 to September 2026)**, which are not in your notebook: nobody's model has seen
them. The fund that turns \$10,000 into the most money wins. Everyone who beats the S&P 500 (SPY) earns a badge.

Then comes the **live round**: you build a second fund on all of the history, it is locked in at Studio 6, and it is
scored on **real prices** through our last class.

## Import modules (Pyomo, ipopt and Bonmin)

Run this first, every time. About a minute.

In [ ]:
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
from pyomo.environ import *

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

# Step 1 · Name your fund and choose the round

Fill in the form and run the cell. Use the `backtest` round tonight; the `live` round opens after the reveal.

In [ ]:
#@title Your fund
fund_name = "My Fund"  #@param {type:"string"}
your_name = "First Last"  #@param {type:"string"}
round_name = "backtest"  #@param ["backtest", "live"]

data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/'
lookup = pd.read_csv(data_folder + 'sp500_lookup.csv')
if round_name == "backtest":
    prices = pd.read_csv(data_folder + 'sp500_prices_past.csv', index_col=0)
else:
    try:
        prices = pd.read_csv(data_folder + 'sp500_prices_full.csv', index_col=0)
    except Exception:
        print("The live round opens after the backtest reveal. Use round_name = 'backtest' for now.")

returns = prices.pct_change().dropna()
risk_ceiling = 0.00223
print("Fund:", fund_name, "  Manager:", your_name, "  Round:", round_name)
print(len(returns), "months of returns, from", returns.index[0], "to", returns.index[-1])
print(len(lookup), "stocks in", lookup['sector'].nunique(), "sectors")

# Step 2 · Pick 3 stocks from each sector

Choose **one filter**. It looks at each sector's stocks and picks 3, using only the history in this notebook:

| Filter | Picks the 3 stocks in each sector with... | The idea |
|---|---|---|
| `momentum` | the best return over the last 12 months | winners keep winning |
| `steady` | the lowest volatility (standard deviation) | calm stocks, fewer bad months |
| `best average` | the best average month over all 48 months | long-run winners |
| `contrarian` | the worst return over the last 12 months | losers bounce back |
| `lucky` | a random draw (seeded by your fund name) | can a dartboard win? |

Then, if you want, **override any sector by hand** in `my_picks`: list exactly 3 tickers for that sector, and the
filter leaves that sector alone. (The table printed below shows every sector's stocks with their numbers.)

In [ ]:
#@title Your filter
my_filter = "momentum"  #@param ["momentum", "steady", "best average", "contrarian", "lucky"]

# optional: hand-pick any sector (exactly 3 tickers), e.g. 'Energy': ['XOM', 'CVX', 'COP'],
my_picks = {
}

# each stock's numbers, from the history in this notebook only
last_12 = (1 + returns.tail(12)).prod() - 1
stats = pd.DataFrame({'last 12 months': last_12, 'volatility': returns.std(), 'average month': returns.mean()})
stats = stats.drop(index='SPY')
stats['sector'] = lookup.set_index('ticker').loc[stats.index, 'sector']
stats['company'] = lookup.set_index('ticker').loc[stats.index, 'company']

chosen = []
seed = 0
for ch in fund_name:
    seed += ord(ch)
random_draw = np.random.default_rng(seed)        # the same fund name always draws the same stocks
for sector in sorted(stats['sector'].unique()):
    in_sector = stats[stats['sector'] == sector]
    if sector in my_picks:
        picks = my_picks[sector]
    elif my_filter == "momentum":
        picks = list(in_sector.sort_values('last 12 months', ascending=False).index[:3])
    elif my_filter == "steady":
        picks = list(in_sector.sort_values('volatility').index[:3])
    elif my_filter == "best average":
        picks = list(in_sector.sort_values('average month', ascending=False).index[:3])
    elif my_filter == "contrarian":
        picks = list(in_sector.sort_values('last 12 months').index[:3])
    else:
        picks = list(random_draw.choice(list(in_sector.index), size=3, replace=False))
    for t in picks:
        chosen.append(t)

my_stocks = stats.loc[chosen, ['sector', 'company', 'last 12 months', 'volatility', 'average month']]
print(len(chosen), "stocks picked")
my_stocks.round(3)

**Check your picks.** 33 stocks, 3 per sector? If you hand-picked a sector, are the tickers spelled right (they
must be in the S&P 500 list)? Run `stats[stats['sector'] == 'Energy'].round(3)` in a new cell to browse a sector.

# Step 3 · Your risk tolerance, then solve

Studio 5's model with the competition rules: every stock you hold gets 2% to 10% (a yes/no $y_k$ for each stock,
with $0.02\,y_k \le x_k \le 0.10\,y_k$), all of the money invested, and the risk at most your limit.

Your risk limit sits between the **safest** your 33 stocks can be (tolerance 0) and the **ceiling** (tolerance 1):

$$\text{risk limit} = \text{safest} + \text{tolerance} \times (\text{ceiling} - \text{safest})$$

In [ ]:
#@title Your risk tolerance
risk_tolerance = 0.8  #@param {type:"slider", min:0, max:1, step:0.05}

df_return = returns[chosen].mean()
df_cov = returns[chosen].cov()


def build(stocks):
    model = ConcreteModel()
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    model.y = Var(stocks, domain=Binary)
    for s in stocks:
        model.x[s] = 1 / len(stocks)
        model.y[s] = 1
    model.constraints = ConstraintList()
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)
    for s in stocks:
        model.constraints.add(model.x[s] <= 0.10 * model.y[s])
        model.constraints.add(model.x[s] >= 0.02 * model.y[s])
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    return model, risk_expr


# the safest your stocks can be under the rules
model, risk_expr = build(chosen)
model.risk = Objective(expr=risk_expr, sense=minimize)
SolverFactory('bonmin', executable=bonmin_path).solve(model)
safest = model.risk()

if safest > risk_ceiling:
    print("Your picks cannot get under the risk ceiling even at their safest (%.5f > %.5f)." % (safest, risk_ceiling))
    print("Try a calmer filter (for example 'steady') or hand-pick calmer stocks in a sector or two.")
else:
    my_limit = safest + risk_tolerance * (risk_ceiling - safest)
    model, risk_expr = build(chosen)
    obj_expr = 0
    for s in chosen:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)
    model.constraints.add(risk_expr <= my_limit)
    results = SolverFactory('bonmin', executable=bonmin_path).solve(model)

    weights = {}
    for s in chosen:
        if model.x[s]() > 0.0005:
            weights[s] = model.x[s]()
    print("safest %.5f, ceiling %.5f -> your risk limit %.5f" % (safest, risk_ceiling, my_limit))
    print("expected (past) average month: %.2f%%, holding %d stocks" % (100 * model.portfolio_return(), len(weights)))
    holdings = pd.Series(weights).sort_values()
    figure(figsize=(8, 7))
    barh(holdings.index, holdings.values, color='tab:orange')
    xlabel('Share of the money')
    title(fund_name + ': what to hold')
    show()

# Step 4 · Submit your fund

This saves your entry as a small CSV file named after your fund and downloads it. **Upload that file to the
Studio 6 competition entry on HuskyCT.** One entry per person per round; the last upload counts.

In [ ]:
entry = pd.DataFrame({'ticker': list(weights.keys()), 'weight': list(weights.values())})
entry['sector'] = lookup.set_index('ticker').loc[entry['ticker'], 'sector'].values
entry['fund_name'] = fund_name
entry['manager'] = your_name
entry['round'] = round_name
entry['filter'] = my_filter
entry['risk_tolerance'] = risk_tolerance
entry['risk_limit'] = my_limit

safe_name = ''
for ch in fund_name:
    if ch.isalnum():
        safe_name += ch
    else:
        safe_name += '_'
file_name = 'entry_' + round_name + '_' + safe_name + '.csv'
entry.to_csv(file_name, index=False)
print("saved", file_name, "- weights add up to %.4f" % entry['weight'].sum())

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(file_name)

### Before you submit, write down

1. Your fund's name, your filter and your risk tolerance.
2. Your three biggest holdings, and why you think the model chose them.
3. **Your prediction:** will your fund beat the S&P 500? Write down what you think your \$10,000 grows to.

Dr. Dave reveals the leaderboard in class, with a race of everyone's \$10,000, month by month.